<a href="https://colab.research.google.com/github/johnxavier10k/fintech-complaints-product-analysis/blob/main/notebooks/02_sql_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mobile Wallet Safety: SQL Analysis

This notebook uses DuckDB SQL to analyze 2025 CFPB mobile and digital wallet complaints, focusing on fraud, unauthorized transactions, company responses, and complaint channels.

In [1]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True
)

Mounted at /content/drive


In [2]:
import duckdb

data_path = "/content/drive/MyDrive/fintech-complaints-product-analysis/fintech_complaints_raw.csv"

con = duckdb.connect()

row_count = con.execute(f"""
    SELECT COUNT(*) AS total_rows
    FROM read_csv_auto('{data_path}', header=True)
""").fetchdf()

row_count

,total_rows
0,84619


## Create the analysis table

The raw CSV contains multiple financial-service categories. This query selects mobile and digital wallet complaints, standardizes column names, converts timestamps, and creates a binary timely-response field.

In [3]:
create_table_sql = f"""
CREATE OR REPLACE TABLE wallet_complaints AS

SELECT
    CAST("Complaint ID" AS BIGINT) AS complaint_id,
    TRY_CAST("Date received" AS TIMESTAMPTZ) AS date_received,
    "Sub-product" AS sub_product,
    "Issue" AS issue,
    "Consumer complaint narrative" AS narrative,
    "Company public response" AS company_public_response,
    "Company" AS company,
    "State" AS state,
    "Submitted via" AS submitted_via,
    TRY_CAST("Date sent to company" AS TIMESTAMPTZ)
        AS date_sent_to_company,
    "Company response to consumer" AS company_response,
    CASE
        WHEN "Timely response?" = 'Yes' THEN 1
        ELSE 0
    END AS is_timely

FROM read_csv_auto('{data_path}', header=True)

WHERE "Sub-product" = 'Mobile or digital wallet';
"""

con.execute(create_table_sql)

In [4]:
validation_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT complaint_id) AS unique_complaints,
    COUNT(*) - COUNT(date_received) AS invalid_received_dates,
    COUNT(*) - COUNT(date_sent_to_company) AS invalid_sent_dates
FROM wallet_complaints;
"""

con.execute(validation_query).fetchdf()

,total_rows,unique_complaints,invalid_received_dates,invalid_sent_dates
0,17150,17150,0,0


## Complaint concentration by issue

This query calculates each issue’s complaint count, share of wallet complaints, and cumulative share. The cumulative percentage helps identify the smallest group of problems responsible for most complaints.

In [5]:
issue_distribution_query = """
WITH issue_counts AS (
    SELECT
        issue,
        COUNT(*) AS complaint_count
    FROM wallet_complaints
    GROUP BY issue
)

SELECT
    issue,
    complaint_count,

    ROUND(
        100.0 * complaint_count
        / SUM(complaint_count) OVER (),
        2
    ) AS complaint_percentage,

    ROUND(
        100.0
        * SUM(complaint_count) OVER (
            ORDER BY complaint_count DESC
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        )
        / SUM(complaint_count) OVER (),
        2
    ) AS cumulative_percentage

FROM issue_counts
ORDER BY complaint_count DESC;
"""

issue_distribution = con.execute(
    issue_distribution_query
).fetchdf()

issue_distribution

,issue,complaint_count,complaint_percentage,cumulative_percentage
0,Unauthorized transactions or other transaction...,5758,33.57,33.57
1,Fraud or scam,4981,29.04,62.62
2,Trouble accessing funds in your mobile or digi...,3038,17.71,80.33
3,"Managing, opening, or closing your mobile wall...",1773,10.34,90.67
4,Confusing or missing disclosures,771,4.50,95.17
5,Confusing or misleading advertising or marketing,290,1.69,96.86
6,Unexpected or other fees,280,1.63,98.49
7,Problem adding money,195,1.14,99.63
8,"Overdraft, savings, or rewards features",64,0.37,100.00


## Monthly wallet safety complaints

This query uses conditional aggregation to compare total wallet complaints with fraud and unauthorized-transaction complaints by month.

In [6]:
monthly_safety_query = """
WITH monthly_counts AS (
    SELECT
        CAST(DATE_TRUNC('month', date_received) AS DATE)
            AS complaint_month,

        COUNT(*) AS total_wallet_complaints,

        SUM(
            CASE
                WHEN issue IN (
                    'Fraud or scam',
                    'Unauthorized transactions or other transaction problem'
                )
                THEN 1
                ELSE 0
            END
        ) AS safety_complaints

    FROM wallet_complaints
    GROUP BY complaint_month
)

SELECT
    complaint_month,
    total_wallet_complaints,
    safety_complaints,

    ROUND(
        100.0 * safety_complaints
        / NULLIF(total_wallet_complaints, 0),
        2
    ) AS safety_complaint_percentage

FROM monthly_counts
ORDER BY complaint_month;
"""

monthly_safety = con.execute(
    monthly_safety_query
).fetchdf()

monthly_safety

,complaint_month,total_wallet_complaints,safety_complaints,safety_complaint_percentage
0,2025-01-01,3455,2363.0,68.39
1,2025-02-01,1319,866.0,65.66
2,2025-03-01,1063,692.0,65.10
3,2025-04-01,1034,650.0,62.86
4,2025-05-01,1062,673.0,63.37
5,2025-06-01,1082,645.0,59.61
6,2025-07-01,1296,766.0,59.10
7,2025-08-01,1347,786.0,58.35
8,2025-09-01,1178,701.0,59.51
9,2025-10-01,1356,821.0,60.55


## Company response outcomes

This query compares company response outcomes for safety complaints against other wallet complaints. Response categories describe how companies closed complaints; they do not measure customer satisfaction or confirm that the underlying problem was resolved.

In [7]:
response_outcome_query = """
WITH categorized_complaints AS (
    SELECT
        CASE
            WHEN issue IN (
                'Fraud or scam',
                'Unauthorized transactions or other transaction problem'
            )
            THEN 'Safety complaint'
            ELSE 'Other wallet complaint'
        END AS complaint_group,

        company_response

    FROM wallet_complaints
),

response_counts AS (
    SELECT
        complaint_group,
        company_response,
        COUNT(*) AS complaint_count

    FROM categorized_complaints
    GROUP BY
        complaint_group,
        company_response
)

SELECT
    complaint_group,
    company_response,
    complaint_count,

    ROUND(
        100.0 * complaint_count
        / SUM(complaint_count) OVER (
            PARTITION BY complaint_group
        ),
        2
    ) AS percentage_within_group

FROM response_counts
ORDER BY
    complaint_group,
    complaint_count DESC;
"""

response_outcomes = con.execute(
    response_outcome_query
).fetchdf()

response_outcomes

,complaint_group,company_response,complaint_count,percentage_within_group
0,Other wallet complaint,Closed with explanation,5729,89.36
1,Other wallet complaint,Closed with non-monetary relief,504,7.86
2,Other wallet complaint,Closed with monetary relief,164,2.56
3,Other wallet complaint,Untimely response,14,0.22
4,Safety complaint,Closed with explanation,9780,91.07
5,Safety complaint,Closed with monetary relief,640,5.96
6,Safety complaint,Closed with non-monetary relief,312,2.91
7,Safety complaint,Untimely response,7,0.07


## SQL findings

- Fraud and unauthorized-transaction issues represented 62.62% of mobile-wallet complaints.
- Including difficulty accessing funds, the top three issues represented 80.33%.
- Safety complaints remained the majority in every month of 2025, ranging from 58.35% to 68.39% of monthly wallet complaints.
- Monetary relief was recorded for 5.96% of safety complaints compared with 2.56% of other wallet complaints.
- Most safety complaints, 91.07%, closed with an explanation.

Company response categories do not measure customer satisfaction, validate the complaint, or indicate whether the consumer recovered the full disputed amount.